# Introduction

## Primary Research Question

To what extent did the imposition of tariffs on Chinese goods (starting July 2018) trigger a Trade Diversion effect, specifically favoring alternative manufacturing hubs like Vietnam and Mexico?

## Hypothesis
We posit that the tariff regime caused a structural break in trade dynamics characterized by:

- Substitution Effect: A significant deceleration or decline in US imports from China, counterbalanced by a surge in imports from Vietnam and Mexico ("Triangulation").

- Decoupling: A weakening of the historical correlation between Chinese and Vietnamese export trends to the US market.

- Data Definition: All trade volumes are analyzed from the perspective of the United States as the Reporting Country. Therefore:

  - "Imports": Refers to US inflows from partner countries (CIF Value).

  - "Exports": Refers to US outflows to partner countries.

# Preparation

## Loadings

In [ ]:
# --- Core Data Analysis ---
import pandas as pd
import numpy as np

# --- Econometrics & Statistics ---
import statsmodels.formula.api as smf
import statsmodels.api as sm

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

# --- Google Colab Utilities ---
from google.colab import files
from google.colab import drive

# --- Settings ---
import warnings
warnings.filterwarnings('ignore')

Google drive for data

importing the datasets we need for the analysis

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


trade: this dataset includes monthly raw import/export from U.S to all other countries, starting from 1987 to 2025.

usd_vnd: contains monthly exchange rate USD/VND (Vietnamese Dong) from 1987 to 2025; for this work, closure data is going to be used to compute YoY metrics.

usd_cny: contains monthly exchange rate USD/CNY (Chinese Yuan) from 1987 to 2025; for this work, closure data is going to be used to compute YoY metrics.

usd_mxn: contains monthly exchange rate USD/MXN (Mexican Pesos) from 1987 to 2025; for this work, closure data is going to be used to compute YoY metrics.

In [ ]:
trade = pd.read_excel("/content/gdrive/MyDrive/CP_project/import_exp_goods_china_us.xlsx")
usd_vnd = pd.read_csv('/content/gdrive/MyDrive/CP_project/USD_VND Historical Data.csv') # in english
usd_cny = pd.read_csv('/content/gdrive/MyDrive/CP_project/USD_CNY Dati Storici.csv', sep=';')
usd_mxn = pd.read_csv('/content/gdrive/MyDrive/CP_project/USD_MXN Dati Storici.csv')

indpro: this dataset contains U.S. monthly industrial production from 1987 to 2025 (november). Data is represented as YoY rate of change computed on seasonally adjusted data.

In [ ]:
indpro  = pd.read_csv('/content/gdrive/MyDrive/CP_project/INDPRO_PC1.csv', parse_dates=['observation_date'], decimal='.', thousands=None)
print(indpro[-5:-1])

Data has been imported. From now on, we have to clean the datasets and merge them in order to perform the analysis. This implies normalizing the formatting of tables.

First of all, it is necessary to work on the dataset "trade": for this work, we are interested in understanding how trading volumes vary between U.S. and China, Mexico and Vietnam, hence all other countries must be removed.

## Data pre-processing

In [ ]:
# trade data
print(trade[1:5])

# we need to select the countries of interest
tgt = ["China", "Mexico", "Vietnam"]

trade_clean = trade[trade['CTYNAME'].isin(["China", "Mexico", "Vietnam"])]

# control
print(trade_clean['CTYNAME'].unique())

Now we have to convert formats (considering the column "year" and other float numbers) into correct ones, moreover it is important to make it into a "long" format.

In [ ]:
# we need to drop totals for import/export
print(trade.columns)

cols_to_drop = ['IYR', 'EYR']
# error='ignore' serve per non bloccare il codice se le hai già tolte
trade = trade.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
# The structure we need: Year, Code, Country, [12 Import], [12 Export]
new_columns = ['year', 'country_code', 'country']

# cols 4-15: import
# generating names like: import_1, import_2, ..., import_12
new_columns.extend([f'import_{i}' for i in range(1, 13)])

# same for export cols
new_columns.extend([f'export_{i}' for i in range(1, 13)])

# controlling number of cols
if len(trade.columns) == len(new_columns):
    trade.columns = new_columns
else:
    print(f"warning: dataset has {len(trade.columns)} columns; {len(new_columns)} expected.")
    print(trade.columns)

# converting in float
cols_to_numeric = [c for c in trade.columns if 'import_' in c or 'export_' in c]
for col in cols_to_numeric:
    trade[col] = pd.to_numeric(trade[col], errors='coerce')

# from wide to long
trade_long = pd.wide_to_long(
    trade,
    stubnames=['import', 'export'],
    i=['year', 'country_code', 'country'],
    j='month',
    sep='_'
).reset_index()

# creating the date -> key for other datasets
trade_long['date'] = pd.to_datetime(
    trade_long['year'].astype(str) + '-' +
    trade_long['month'].astype(str) + '-01'
)

# ordering for country, date and renaming
trade_long = trade_long.sort_values(['country', 'date'])

trade_long = trade_long.rename(columns={'import': 'import_value', 'export': 'export_value'})

# re-order
trade_clean = trade_long[['date', 'year', 'month', 'country', 'country_code', 'import_value', 'export_value']]

# focusing on selecting the variables we need
# we need to select the countries of interest
tgt = ["China", "Mexico", "Vietnam"]

trade_final = trade_clean[trade_clean['country'].isin(["China", "Mexico", "Vietnam"])]

# control
print(trade_final['country'].unique())

print(trade_final.head())

print("\n --- \n")
print(trade_final.info())

In [ ]:
trade_final.to_csv('Dataset_Finale.csv', index=False, sep=';', decimal=',')

print(trade_final[1:3])
# files.download('Dataset_Finale.csv')

Formatting and merging all datasets into one master dataset.

In [ ]:
# building a function to clean datasets to merge with the master one

def clean_rate(rate_df, country_name):
  # create a copy of df
  df_copy = rate_df.copy()

  # dfs have different formats
  if country_name in ['China', 'Mexico']:

    # date format
    df_copy['date'] = pd.to_datetime(df_copy['Data'], format='%d.%m.%Y')

    # exchange rate format
    df_copy['exchange_rate'] = df_copy['Ultimo'].astype(str).str.replace(',', '.').astype(float)

    # adds country col
    df_copy['country'] = country_name

  else:

    # date format
    df_copy['date'] = pd.to_datetime(df_copy['Date'], format='%m/%d/%Y')

    # exchange rate format
    df_copy['exchange_rate'] = df_copy['Price'].astype(str).str.replace(',', '').astype(float)

    # adds country col
    df_copy['country'] = country_name

  return df_copy[['date', 'country', 'exchange_rate']]

# print(clean_rate(usd_mxn, 'Mexico')[1:3])
# print(clean_rate(usd_vnd, 'Vietnam')[1:3])

usd_cny_clean = clean_rate(usd_cny, 'China')
usd_mxn_clean = clean_rate(usd_mxn, 'Mexico')
usd_vnd_clean = clean_rate(usd_vnd, 'Vietnam')

print(usd_vnd_clean.info())
print("\n   -----   \n")
print(usd_cny_clean.info())

print(usd_mxn_clean.info())

In [ ]:
def clean_indpro(df):
    df_copy = df.copy()

    # rename
    df_copy = df_copy.rename(columns={'observation_date': 'date'})
    df_copy['date'] = pd.to_datetime(df_copy['date'])
    return df_copy

indpro_clean = clean_indpro(indpro)
print(indpro_clean.info())

Now, we can merge information all together.

In [ ]:
usd_cny_clean['country'] = 'China'
usd_vnd_clean['country'] = 'Vietnam'
usd_mxn_clean['country'] = 'Mexico'

# merge rate dfs
all_rates = pd.concat([usd_cny_clean, usd_vnd_clean, usd_mxn_clean], ignore_index=True)

# joining with master df -> using inner we only choose records where i have all data
trade_master = pd.merge(
    trade_final,
    all_rates,
    on=['date', 'country'],
    how='inner'
)

# same with indpro
trade_master = pd.merge(
    trade_master,
    indpro_clean,
    on='date',
    how='inner'
)

trade_master = trade_master.sort_values(by=['country', 'date']).reset_index(drop=True)

# res
print(f"final dimension of df {trade_master.shape}")
print(f"starting from {trade_master['date'].dt.date.min()} to {trade_master['date'].dt.date.max()}")
print("-" * 50)

# NAs control
if trade_master.isnull().sum().sum() == 0:
    print("Success! All data are available")
else:
    print("Warning!")
    print(trade_master.isnull().sum())

display(trade_master.head())

In [ ]:
# to reload it
# df = pd.read_csv('trade_master.csv', sep=';', decimal=',')

## Feature Engineering

For this project, we aim to prepare data even better. Before performing the econometric analysis, the dataset needs specific transformations to ensure statistical robustness and economic interpretability.

These feature engineering steps—Logarithmic transformation, Structural Break identification, and Time Lags—are standard practices in international trade econometrics.

- Logarithmic Transformation (Elasticity): We transformed the dependent variable (Imports) and key independent variables (Industrial Production as Health proxy, Exchange Rate) into natural logarithms. In international economics, we are rarely interested in absolute unit changes (e.g., "Imports increased by $10 million"). Instead, we focus on elasticity: the percentage change in Y given a percentage change in X. This transformation linearizes the relationship between variables and allows us to interpret the regression coefficients directly as percentages. For example, a coefficient of -1.5 on the exchange rate would imply that a 1% appreciation of the currency leads to a 1.5% decrease in imports.

- The "Trade War" Dummy Variable (Structural Break): To test the hypothesis of trade diversion, we introduced a binary dummy variable named trade_war that takes the value 0 for the period prior to July 2018 and 1 for the subsequent period. July 2018 marks the implementation of the first major tranche of Section 301 tariffs by the US administration against China. This variable acts as a "treatment" indicator, allowing us to statistically test if there was a significant structural break in trade patterns specifically triggered by this policy shift, distinguishing it from normal long-term trends.

- Lagged Variables (Temporal Dynamics): We introduced a 3-month lag (lag_3) for the Exchange Rate variable. International trade is characterized by "stickiness" Supply contracts are often signed months in advance, meaning that import volumes do not react instantly to daily currency fluctuations. By using the exchange rate from three months prior (t-3) to explain the imports of the current month (t), the model captures the realistic delay in the economic transmission mechanism (the J-Curve effect), resulting in a more accurate estimation of price sensitivity.

In [ ]:
# FEATURE ENGINEERING

# control on date
trade_master['date'] = pd.to_datetime(trade_master['date'])

# log transformation

trade_master['ln_import'] = np.log(trade_master['import_value'] + 1)
trade_master['ln_export'] = np.log(trade_master['export_value'] + 1)
trade_master['ln_inpdro'] = np.log(trade_master['INDPRO_PC1'])
trade_master['ln_exchange'] = np.log(trade_master['exchange_rate'])

# 3. trade war dummy variable
# value 1 for records after july 2018; else 0
trade_master['trade_war'] = trade_master['date'].apply(lambda x: 1 if x >= pd.Timestamp('2018-07-01') else 0)

# 4. lags
# we need to sort values and group country in order to shift correctely
trade_master = trade_master.sort_values(by=['country', 'date'])

trade_master['ln_exchange_lag3'] = trade_master.groupby('country')['ln_exchange'].shift(3)

# final cleaning
trade_master = trade_master.dropna().reset_index(drop=True)

print(f"Updated dataset: {trade_master.shape}")
print("-" * 50)
print("Full dataset")
print(trade_master.head())

In [ ]:
print(trade_master[(trade_master['year'] == 2025) & (trade_master['month'] == 12)])

# we need to remove month 12 as data are not fully complete
trade_master = trade_master.drop(trade_master[(trade_master['year'] == 2025) & (trade_master['month'] == 12)].index)

In [ ]:
print(trade_master[(trade_master['year'] == 2025) & (trade_master['month'] == 12)])

trade_master.to_csv('trade_master.csv', index=False, sep=';', decimal=',')
# files.download('trade_master.csv')

# Exploratory Data Analysis (EDA)

In this chapter we are going to perform EDA through three plots:

- A: Import Growth over years with 2018 as baseline (post US tariffs);

- B: Boxplot representing the distribution of log-import in US before and after tariffs application;

- C: Rolling Correlation between China and Vietnam/Mexico.

Through this analysis we want to delve into the core role of China as an exporter of (mainly) goods to US and how it relates to the increasing import volumes from Vietnam and Mexico; one central hypothesis of this work is that these countries act as strategic intermediaries, as Chinese goods may re-routed or assembled in these territories before entering the US market, hence bypassing specific tariffs.

At first, we need to prepare data for plot visualization.

In [ ]:
sns.set_theme(style="whitegrid")

# wide format
trade_wide = trade_master.pivot(index='date', columns='country', values='import_value')

# computing Rolling Correlation -> correlation in a specific time frame
corr_vietnam = trade_wide['China'].rolling(window=24).corr(trade_wide['Vietnam'])
corr_mex = trade_wide['China'].rolling(window=24).corr(trade_wide['Mexico'])

# creating index in base 2018 (tariffs where set in this year)
base_date = '2018-01-01'

base_values = trade_wide.loc[base_date] # takes record corresponding to base_date (.loc selecting by index label)
df_indexed = trade_wide.div(base_values) * 100 # takes all records and divide them by the value at base_date

## A: Import Growth over years with 2018 as baseline (post US tariffs)

To compare the trade performance of China against Vietnam and Mexico, we must address the issue of scale heterogeneity. China’s absolute export volume to the US is significantly larger than that of Vietnam. Comparing raw values would mask the relative growth dynamics. Therefore, we applied a normalization technique, indexing all import values to 100 starting from January 2018 (the baseline period immediately preceding the major tariff escalations).

What this graph shows This visualization is designed to highlight divergence.

The Baseline: The horizontal line at 100 represents the status quo at the beginning of 2018.

In [ ]:
# A: Import Growth over years with 2018 as baseline (post US tariffs)

df_pivot = trade_master.pivot(index='date', columns='country', values='import_value')
base_values = df_pivot.loc['2018-01-01']
df_indexed = (df_pivot.div(base_values) * 100).reset_index()


plot_data = df_indexed.copy()

# plot visualization
fig = go.Figure()

colors = {'China': '#d62728',    # red
          'Vietnam': '#2ca02c',  # green
          'Mexico': '#1f77b4'}   # blue

for country in ['China', 'Vietnam', 'Mexico']:
    fig.add_trace(go.Scatter(
        x=plot_data['date'],
        y=plot_data[country],
        mode='lines',
        name=country,
        line=dict(color=colors[country], width=2.5),
        hovertemplate=f'<b>{country}</b>: Index %{{y:.1f}}<br>Date: %{{x|%b %Y}}<extra></extra>'
    ))

# tariffs start
fig.add_vline(
    x=pd.Timestamp('2018-07-01').timestamp() * 1000,
    line_width=2, line_dash="dash", line_color="black",
    annotation_text="US Tariffs Start",
    annotation_position="top right"
)

fig.add_hline(
    y=100,
    line_width=1, line_dash="dot", line_color="gray",
    annotation_text="Baseline (Jan 2018 = 100)",
    annotation_position="bottom right"
)

fig.update_layout(
    title=dict(
        text='<b>A. Evidence of Trade Diversion: Import Growth Trends</b><br><sup>(Base 100 = Jan 2018)</sup>',
        font=dict(size=20)
    ),
    xaxis_title='Date',
    yaxis_title='Growth Index (Jan 2018 = 100)',
    template='plotly_white',
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

# download
# fig.write_html("plot_A_trade_diversion_interactive.html")
# files.download("plot_A_trade_diversion_interactive.html")

The "Scissors Effect": as the Chinese trend were slowly increasing over years, after the red line it starts to fall, showing a little peak at years around 2021/2022 and then strongly declining in correspondence of the latest announced tariffs. On the other hand, Vietnam is truly living an expansion phase, reaching a very high level in 2025, suggesting that importers shifted their sourcing away from China to alternative partners.  Mexico is instead showing an increasing trend that seems almost constant thrugh time and not so influenced by tariffs; this is a representation of a dual aspect: firstly, what is called nearshoring: US understood that relocating production to a neighboring country is less risky than doing it to a geographically farther one; secondly, it is due to commercial agreements between US and Mexico.

In [ ]:
# B. Boxplot representing the distribution of log-import in US before and after tariffs application

# taking 6 years before tariffs to make a symmetrical comparison

start_date_comparison = '2012-01-01'
df_box = trade_master[trade_master['date'] >= pd.Timestamp(start_date_comparison)].copy()

df_box['Scenario'] = df_box['trade_war'].map({0: 'Pre-War (2012-2018)', 1: 'Post-War (2018-2025)'})

# plot visualization
fig = go.Figure()

colors = {'Pre-War (2012-2018)': '#87CEEB',  # SkyBlue
          'Post-War (2018-2025)': '#FA8072'} # Salmon

# creating the groups: pre-war vs post-war
for scenario in ['Pre-War (2012-2018)', 'Post-War (2018-2025)']:
    subset = df_box[df_box['Scenario'] == scenario]

    fig.add_trace(go.Box(
        y=subset['ln_import'],
        x=subset['country'],
        name=scenario,
        marker_color=colors[scenario],
        boxmean=True,
        boxpoints='outliers'
    ))

fig.update_layout(
    boxmode='group',
    title=dict(
        text='<b>Robustness Check: Import Distribution Shift</b><br><sup>Symmetric Comparison (2012-2018 vs 2018-2025)</sup>',
        font=dict(size=20)
    ),
    yaxis_title='Log(Import Value)',
    xaxis_title='Country',
    template='plotly_white',
    legend=dict(yanchor="bottom", y=0.1, xanchor="left", x=0.01)
)

fig.show()

# download
# fig.write_html("boxplot_robustness_interactive.html")
# files.download("boxplot_robustness_interactive.html")

What stands out here is how the countries differ in terms of "Pre-War" and "Post-War" transition: China is on a very high level - as one of the top exporters in the world - but the red boxplot shows that, after 2018, the volumes distribution is wider, suggesting that imports tend to reach minimum levels never reached before; moreover, the two boxes are on a similar level, hence there's no actual growth with respect to the past.

Mexico and Vietnam reach two very different level on the two eras: this is particularly true for Vietnam, where the detach is clear, and levels of imports grow exponentially.

## C. Rolling Correlation between countries

In a globalized supply chain, emerging economies often move in synchronization with major manufacturing hubs. Historically, when US demand increased, imports from both China and Vietnam would rise together. However, a Trade War introduces a substitution effect. To test this hypothesis, we employ a 24-Month Rolling Correlation analysis. This metric tracks how the relationship between Chinese and Vietnamese exports to the US has evolved over time.

Rolling Correlation: as standard correlation analysis provides a single summary statistic (e.g., Pearson’s r) representing the relationship between two variables over the entire timeframe, here we apply a dynamic technique that calculates the correlation coefficient within a moving window of a fixed size (in our case, 24 months). This window slides forward one month at a time. By doing so, this method allows us to detect structural breaks and regime shifts. It answers the question: 'Is the economic synchronization between China and Vietnam stable, or has it deteriorated following the Trade War?' A dropping curve indicates decoupling, where the two economies no longer move together.


In [ ]:
# C. Rolling Correlation between countries

df_pivot = trade_master.pivot(index='date', columns='country', values='import_value')

# rolling correlation
window = 24
corr_vietnam = df_pivot['China'].rolling(window=window).corr(df_pivot['Vietnam'])
corr_mex = df_pivot['China'].rolling(window=window).corr(df_pivot['Mexico'])

# plot visualization
fig = go.Figure()

# China/Vietnam
fig.add_trace(go.Scatter(
    x=corr_vietnam.index,
    y=corr_vietnam,
    mode='lines',
    name='Corr: China vs Vietnam',
    line=dict(color='purple', width=3),
    hovertemplate='<b>China-Vietnam</b>: %{y:.2f}<br>Date: %{x|%b %Y}<extra></extra>'
))

# China/Mexico
fig.add_trace(go.Scatter(
    x=corr_mex.index,
    y=corr_mex,
    mode='lines',
    name='Corr: China vs Mexico',
    line=dict(color='green', width=2.5, dash='dash'), # Tratteggiata come nel tuo originale
    hovertemplate='<b>China-Mexico</b>: %{y:.2f}<br>Date: %{x|%b %Y}<extra></extra>'
))


# Start tariffs
fig.add_vline(
    x=pd.Timestamp('2018-07-01').timestamp() * 1000,
    line_width=2, line_dash="dot", line_color="red",
    annotation_text="Start of Trade War",
    annotation_position="top right"
)

fig.add_hline(
    y=0,
    line_width=1.5, line_color="black",
    annotation_text="Decoupling Point (Zero Correlation)",
    annotation_position="bottom right"
)

fig.update_layout(
    title=dict(
        text='<b>C. Decoupling Evidence: Who Replaced China?</b><br><sup>Rolling Correlation (24-months window)</sup>',
        font=dict(size=20)
    ),
    xaxis_title='Date',
    yaxis_title='Correlation Coefficient (-1 to +1)',
    template='plotly_white',
    hovermode="x unified", # Compara le due correlazioni passando il mouse

    legend=dict(
        yanchor="bottom",
        y=0.01,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.8)" # Sfondo semitrasparente per leggere sopra le linee
    )
)

fig.show()

# download
# fig.write_html("plot_C_correlation_interactive.html")
# files.download("plot_C_correlation_interactive.html")

Several years (approximately from 2000 to 2018) show a high positive correlation between both China and Vietnam, China and Mexico. This suggest globalization and similar imports volumes for US from all the three countries.

After 2018 the curves fall drastically, becoming even negative for certain timesteps: this implies that, for instance, as China loses market share, Vietnam gains it. This inverse relationship is the mathematical signature of trade diversion.

In [ ]:
# summary

print("\n--- key statistics ---")
summary = trade_master.groupby(['country', 'trade_war'])['import_value'].mean().unstack()
summary.columns = ['Pre-2018 (Avg $M)', 'Post-2018 (Avg $M)']
summary['Growth (%)'] = ((summary['Post-2018 (Avg $M)'] - summary['Pre-2018 (Avg $M)']) / summary['Pre-2018 (Avg $M)']) * 100
display(summary.round(2))

# Data Modeling

## Linear Regression for countries

Here, the goal is to  quantify the extent of Trade Diversion and isolate the specific impact of tariffs from other macroeconomic drivers. To do so, we employ an Ordinary Least Squares (OLS) regression model.

The objective is to test for a structural break in the import function. We model US imports from each partner country (China, Vietnam, Mexico) as a function of US domestic demand, currency competitiveness, and the dummy variable representing the trade war era. This approach allows us to estimate the ceteris paribus effect of the tariff regime: specifically, how much import growth (or contraction) is attributable solely to the post-2018 policy shift, holding economic growth and exchange rates constant.

We estimate the following log-linear equation for each trading partner:


$$\ln(\text{Import}_t) = \beta_0 + \beta_1 ln(\text{US_Demand}_t) + \beta_2 \ln(\text{Exchange_Rate}_t) + \beta_3 (\text{TradeWar_Dummy}_t) + \epsilon_t$$


Where:

$\ln(\text{Import}_t)$: Natural log of monthly import values (Real flow).

$\ln(\text{US_Demand}_t)$: Natural log of the US Industrial Production Index (INDPRO). This serves as a proxy for US economic health and absorption capacity.

$\ln(\text{Exchange_Rate}_t)$: Natural log of the bilateral exchange rate.

$\text{TradeWar_Dummy}_t$: A binary variable taking the value of 0 for the pre-tariff period (before July 2018) and 1 for the post-tariff period.

Where $\epsilon_t$ represents the error term. Given the likely presence of serial correlation and heteroscedasticity in monthly trade time series data, we used Robust Standard Errors (HC3) to ensure statistical validity and prevent biased t-statistics.

In [ ]:
# OLS linear regression
def run_structural_break_model(country_name, df):
    print(f"\n{'='*20} Regression for: {country_name.upper()} {'='*20}")

    # filtering
    df_country = df[df['country'] == country_name].copy()

    # ln_import = response (Y)
    # ln_inpdro = US demand (proxy: industrial production) (x)
    # ln_exchange = exchange rate -> (x)
    # trade_war = DUMMY (0 o 1)
    equation = "ln_import ~ ln_inpdro + ln_exchange + trade_war"

    # execute OLS and solving for heteroschedasiticity
    model = smf.ols(formula=equation, data=df_country).fit(cov_type='HC3')

    # results
    print(model.summary())

    return model

model_china = run_structural_break_model('China', trade_master)
model_vietnam = run_structural_break_model('Vietnam', trade_master)
model_mexico = run_structural_break_model('Mexico', trade_master)

# interpretation
print("\n" + "="*50)
print("COEFFICIENTS 'trade_war':")
print(f"China:    {model_china.params['trade_war']:.4f} (P-value: {model_china.pvalues['trade_war']:.4f})")
print(f"Vietnam: {model_vietnam.params['trade_war']:.4f} (P-value: {model_vietnam.pvalues['trade_war']:.4f})")
print(f"Mexico: {model_mexico.params['trade_war']:.4f} (P-value: {model_mexico.pvalues['trade_war']:.4f})")
print("="*50)
print("Legend:")
print("- if coef is positive (> 0) and P-value < 0.05 -> Trade Diversion")
print("- if coef is negative (< 0) -> Tariffs damage")

$\beta_1$ (Income Elasticity of Demand): This coefficient measures the sensitivity of imports to the US economy. A positive sign ($\beta_1 > 0$) indicates that as US industrial activity rises, demand for intermediate and capital goods from the partner country decreases. (Example: If $\beta_1 = -1.5$, a 1% increase in US production leads to a -1.5% decrease in imports).

$\beta_2$ (Price Competitiveness / Exchange Rate): This captures the effect of currency fluctuations. A positive coefficient typically implies that a depreciation of the partner's currency (making their goods cheaper in USD) leads to higher export volumes to the US.

$\beta_3$ (The Structural Break / Trade War Effect): This is the coefficient of primary interest. It measures the percentage shift in the intercept of the import function during the trade war period, net of demand and currency effects.If $\beta_3 < 0$ and it is significant, it provides statistical evidence of Trade Destruction. The country has systematically lost market access due to the tariffs (Expected for China). On the other hand, if $\beta_3 > 0$ and significant, it provides statistical evidence of Trade Diversion. The country has experienced "abnormal" growth not explained by US GDP or currency, confirming its role as a substitute supplier (expected for Vietnam and Mexico).

The OLS regression analysis yields statistically significant results for all three trading partners ($p < 0.001$), confirming the hypothesis of a structural break in trade patterns following the implementation of Section 301 tariffs in July 2018.

Vietnam: the model provides the strongest empirical evidence of Trade Diversion for Vietnam.The Trade War Effect ($\beta_{war} = 0.418$): The coefficient is positive and highly significant. This indicates that, holding US demand and exchange rates constant, imports from Vietnam have shifted upwards by approximately 41.8% during the trade war period compared to the pre-2018 average. This is a massive structural shift, confirming Vietnam as the primary substitute for manufacturing capacity relocation.
Currency Sensitivity ($\beta_{fx} = 7.06$): The extremely high coefficient for the exchange rate suggests that Vietnamese exports are hyper-sensitive to price competitiveness. A depreciation of the Vietnamese Dong acts as a powerful multiplier for exports to the US.

Mexico (nearshoring effect): Mexico shows a similar, but more moderate, positive trend, consistent with the "Nearshoring" narrative. The Trade War Effect ($\beta_{war} = 0.138$): The dummy variable coefficient is positive and significant. The trade war era has contributed to a structural increase in imports of approximately 13.9%.
While less explosive than Vietnam's growth, this result confirms that Mexico has successfully absorbed a portion of the supply chain, likely favored by the geographical proximity and the USMCA framework.

The result for China ($\beta_{war} = 1.335$) requires a nuanced interpretation to avoid misreading the "Trade Destruction" hypothesis. Why is it positive? The coefficient is positive because the "Post-War" period (2018–2025) is compared against a "Pre-War" baseline that  extends back decades (e.g., to 1987).
Despite the recent stagnation caused by tariffs, the absolute volume of trade today remains historically higher than the long-term average of the last 30 years.

The Signal of Decoupling: The key insight lies in the US Demand Elasticity ($\beta_{demand} = -0.265$). The negative sign is an economic anomaly (usually, when US GDP grows, imports grow). This negative correlation suggests a disconnection: as the US industrial engine grows, it is no longer pulling Chinese imports along with it, hinting that the "transmission mechanism" has indeed broken.

In the end, the econometric evidence supports the Trade Diversion hypothesis. The dummy variable coefficients for Vietnam and Mexico confirm that the post-2018 period represents a structural regime shift, with 'abnormal' growth rates inexplicable by simple demand dynamics.
Conversely, while China maintains high historical volumes (scale effect), its negative elasticity to US industrial production suggests a functional decoupling: the US economy is growing without dragging Chinese imports with it, having substituted them with flows from Vietnam and Mexico.

## Slope-Shifting Model

Beyond quantifying the absolute change in trade volumes, it is crucial to understand if the fundamental economic relationship between the US and its partners has altered. The standard OLS model assumes that the sensitivity of imports to US demand (income elasticity) remains constant over time. However, the 'Decoupling' hypothesis suggests that the Trade War may have structurally weakened the link between US economic growth and Chinese exports.To test this, we extend the baseline model by introducing an interaction term between US Industrial Production (ln_inpdro) and the Trade War dummy. This allows the slope of the demand curve to vary across the two periods, effectively testing if the 'engine' of trade transmission has been modified.

The Model & Coefficients: the equation is defined as:

$$\ln(Import_t) = \beta_0 + \beta_1 \ln(IndPro_t) + \beta_2 (\text{TradeWar}_t) + \beta_3 (\ln(IndPro_t) \times \text{TradeWar}_t) + \epsilon_t$$

Key Interpretation of $\beta_3$ (The Interaction Term): The coefficient $\beta_3$ represents the change in elasticity during the Trade War period. The total elasticity in the post-2018 era becomes $(\beta_1 + \beta_3)$.

Case A: The Decoupling Signal: If $\beta_3 < 0$ and significant, it implies that Chinese exports have become less responsive to US economic activity. Even if the US economy grows, it pulls in fewer Chinese goods than it would have in the past. This is statistical evidence of structural decoupling.

Case B: Enhanced Integration (Possible for Vietnam/Mexico)If $\beta_3 > 0$ and significant, it implies a strengthening of the economic bond. The partner country has become more sensitive to US demand cycles, suggesting deeper supply chain integration.

Case C: Stability (Null Hypothesis) If $\beta_3$ is not statistically significant ($p > 0.05$), it indicates that while the level of trade may have changed (intercept shift), the underlying responsiveness to US demand remains unchanged.

In [ ]:
def run_interaction_model(country_name, df):
    print(f"\n{'='*20} Structural analysis: {country_name.upper()} {'='*20}")

    df_country = df[df['country'] == country_name].copy()

    # interaction term
    # it explains how production change based on the period before/after 2018
    # Formula: Y = b0 + b1*indpro + b2*Dummy + b3*(indpro*Dummy)

    # Se b3 è NEGATIVO e significativo -> Il paese è diventato "sconnesso" dall'economia USA.

    equation = "ln_import ~ ln_inpdro * trade_war + ln_exchange"

    model = smf.ols(formula=equation, data=df_country).fit(cov_type='HC3')

    print(model.summary())

    interaction_coef = model.params['ln_inpdro:trade_war']
    p_val = model.pvalues['ln_inpdro:trade_war']

    print(f"\n--- key result (Change in Elasticity) ---")
    print(f"interaction coefficient: {interaction_coef:.4f}")
    print(f"p-value: {p_val:.4f}")

    if p_val < 0.05:
        if interaction_coef < 0:
            print("Negative structural break: relation with US became weaker.")
        else:
            print("Positive structural break: relation with US became stronger.")
    else:
        print("No structural change.")

run_interaction_model('China', trade_master)
run_interaction_model('Vietnam', trade_master)

Before interpreting the change, we must address the negative baseline coefficients for US Industrial Production (indpro) (-0.43 for China, -0.28 for Vietnam). Historically (1987–2018), trade followed an "Offshoring Dynamic". As US domestic manufacturing stagnated or declined (lower INDPRO), imports increased. They were substitutes, not complements. The US bought from abroad instead of making at home

Vietnam: The interaction term is positive and significant ($p < 0.001$). It effectively neutralizes the historical negative slope ($-0.28 + 0.31 \approx +0.03$). This confirms Supply Chain Integration. Post-2018, Vietnam is no longer just a destination for offshoring (replacing US factories). It has become a vital link in the US industrial machine. Now: When US factories ramp up production, they need components from Vietnam. The relationship has turned pro-cyclical. Vietnam has successfully plugged itself into the US business cycle.

China: Surprisingly, China shows a massive positive shift in elasticity, flipping the total slope to positive ($-0.43 + 0.60 \approx +0.17$). This does not mean China is "winning" more, but that it has lost its "Super-Growth" status.

**Pre-2018**: Chinese exports grew regardless of the US economy (driven by low costs and structural displacement).

**Post-2018**: China has been "demoted" to a standard trading partner. Its exports now only grow if the US economy pulls them (positive elasticity). This is evidence of "Conditional Decoupling": China is no longer a structural displacer of US industry, but more of a residual supplier that serves US demand when domestic capacity is maxed out.

The Structural Break analysis confirms a regime change for both partners. The positive significant interaction terms indicate that the era of 'Offshoring' (where US decline fueled Import growth) is statistically over.
For Vietnam, this signals maturity: it has evolved from a marginal player to an integrated partner whose fortunes are now tied to US industrial health.
For China, it signals a loss of structural momentum: its exports are no longer propelled by an infinite displacement of US manufacturing, but are now constrained to follow the fluctuations of US demand, validating the intent of the trade barriers to normalize the playing field.

## Simulation on Vietnam earnings after 2018 without tariffs

While the regression coefficients confirm the statistical significance of the structural break, they remain abstract parameters. To fully grasp the economic magnitude of the Trade Diversion, we move from inference to simulation.

In this final step, we construct a simulation scenario:

'***What would US imports from Vietnam have looked like if the Trade War had never happened?***'

We use the parameters estimated in the OLS model to predict two distinct trajectories for the post-2018 period:

- The Actual Trajectory (Blue Line): The model prediction using real-world data, including the active Trade War dummy ($D=1$).

- The Counterfactual Baseline (Red Line): A hypothetical simulation where the Trade War dummy is forced to zero ($D=0$), while holding all other macroeconomic variables (US Demand and Exchange Rates) at their actual historical levels.

The vertical distance between the two lines represents the 'Tariff Premium': the specific volume of trade that Vietnam gained solely due to the tariff-induced relocation of supply chains, net of natural organic growth.

In [ ]:
# Counterfactual analysis on Vietnam

df_vietnam = trade_master[trade_master['country'] == 'Vietnam'].copy()
model_cf = smf.ols("ln_import ~ ln_inpdro + ln_exchange + trade_war", data=df_vietnam).fit()

# hypothesis: trade_war is always 0 (tariffs never happened)
df_vietnam['scenario_actual'] = df_vietnam['trade_war'] # 0,1
df_vietnam['scenario_counterfactual'] = 0               # always 0

# tariff prediction
df_vietnam['log_pred_actual'] = model_cf.predict(df_vietnam[['ln_inpdro', 'ln_exchange', 'trade_war']])

# counterfactual prediction (without tariffs)
X_cf = df_vietnam[['ln_inpdro', 'ln_exchange']].copy()
X_cf['trade_war'] = 0
df_vietnam['log_pred_cf'] = model_cf.predict(X_cf)

# converting: exp(ln(x)) = x
df_vietnam['Real_Import_Actual'] = np.exp(df_vietnam['ln_import']) # true data
df_vietnam['Real_Import_Model']  = np.exp(df_vietnam['log_pred_actual']) # predicted values
df_vietnam['Real_Import_NoWar']  = np.exp(df_vietnam['log_pred_cf'])    # predicted values without tariffs

# computing the difference
df_vietnam['Trade_Diversion_Bonus'] = df_vietnam['Real_Import_Actual'] - df_vietnam['Real_Import_NoWar']

# interactive plot
fig = go.Figure()

# 1. red line
fig.add_trace(go.Scatter(
    x=df_vietnam['date'],
    y=df_vietnam['Real_Import_NoWar'],
    mode='lines',
    name='Counterfactual (No Trade War)',
    line=dict(color='firebrick', width=2, dash='dash'), # Linea tratteggiata rossa
    hovertemplate='%{y:,.0f} M$ (Theoretical)<extra></extra>' # Format del tooltip
))

# real data
fig.add_trace(go.Scatter(
    x=df_vietnam['date'],
    y=df_vietnam['Real_Import_Actual'],
    mode='lines',
    name='Actual Imports (Real Data)',
    line=dict(color='navy', width=3), # Linea solida blu
    fill='tonexty', # RIEMPIE L'AREA fino alla traccia precedente (quella rossa)
    fillcolor='rgba(50, 205, 50, 0.2)', # Verde lime semi-trasparente
    hovertemplate='%{y:,.0f} M$ (Actual)<extra></extra>'
))

# 2018 line
fig.add_vline(
    x=pd.Timestamp('2018-07-01').timestamp() * 1000,
    line_width=2,
    line_dash="dot",
    line_color="black",
    annotation_text="Trade War Start",
    annotation_position="top left"
)

fig.update_layout(
    title=dict(
        text='<b>The Vietnam Bonus: Actual vs. Counterfactual (Interactive)</b>',
        font=dict(size=20)
    ),
    xaxis_title='Date',
    yaxis_title='Monthly Import Value (Millions USD)',
    template='plotly_white', # Sfondo bianco pulito accademico
    hovermode="x unified",   # Mostra entrambi i valori quando passi il mouse su una data
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()

# save
# fig.write_html("vietnam_counterfactual_interactive.html")
# files.download("vietnam_counterfactual_interactive.html")

The plot presents a counterfactual simulation based on the OLS model coefficients.

The red dashed line represents the estimated trajectory of Vietnamese imports in the absence of the trade war (holding US demand and exchange rates at their actual levels). The divergence between the solid blue line (actual reality) and the red line (counterfactual) visualizes the magnitude of the Trade Diversion Effect. We estimate that since July 2018, the 'Tariff Premium' has generated approximately $145 Billion in additional export revenue for Vietnam, representing the tangible value of supply chain relocation.

# Conclusion

This study provides robust empirical evidence supporting the Trade Diversion hypothesis following the 2018 US-China trade war. Through a combination of descriptive analysis and structural econometric modeling, we identified three key findings:

1. Divergence in Trade Flows: Visual inspection and rolling correlations confirm a decoupling between US-China trade and a simultaneous surge in imports from Vietnam and Mexico. The historical synchronization between Chinese and Vietnamese exports has broken, shifting towards a substitutionary relationship.

2. Quantification of the 'Vietnam Effect': The econometric model estimates a structural increase in Vietnamese imports of approximately 41.8% in the post-tariff period, net of demand and currency effects. This confirms Vietnam as the primary beneficiary of supply chain relocation.

3. Structural Break in Transmission: The interaction analysis reveals a fundamental shift in economic integration. While Chinese exports have become less responsive to US growth dynamics (conditional decoupling), Vietnam has demonstrated a deepened cyclical integration with the US economy.

In summary, the data suggests that while tariffs successfully reduced direct reliance on China, they did not necessarily repatriate manufacturing to the US. Instead, they catalyzed a reorganization of the global value chain, redirecting demand towards alternative low-cost hubs (Vietnam) and nearshoring partners (Mexico), effectively validating the 'Triangulation' phenomenon.